# Dual-GPU NCCL validation

**EXPERIMENTAL — KAGGLE DUAL-T4 VALIDATION PENDING.** Not executed by the author; outputs are produced only when *you* run it.

Accelerator: **GPU T4 x2**, Internet on only if the model must be downloaded. Run notebooks in order 01 → 06; each stage writes JSON to `/kaggle/working/stokastisk-acceptance/`.

Real `torchrun` 2-rank all-reduce / all-gather / reduce-scatter with process-group teardown. NCCL_DEBUG logs are saved so the actual transport (SHM vs sockets) can be read; nothing here assumes P2P.

In [ ]:
import glob, os, pathlib
# Locate the release bundle (the Kaggle dataset holding manifests/SHA256SUMS). Override with STOKASTISK_BUNDLE.
hits = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
BUNDLE = pathlib.Path(os.environ.get("STOKASTISK_BUNDLE") or (hits[0] if hits else "")) 
assert (BUNDLE / "manifests" / "SHA256SUMS").is_file(), "attach the stokastisk-0.3.0 release dataset to this notebook"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("bundle:", BUNDLE)

In [ ]:
import os
env = dict(os.environ, NCCL_DEBUG="INFO", NCCL_DEBUG_FILE="/kaggle/working/stokastisk-acceptance/nccl.%h.%p.log",
           PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", ""))
import subprocess
r = subprocess.run(["torchrun", "--standalone", "--nproc-per-node=2", str(SCRIPTS / "04_nccl_dual_gpu.py")], env=env)
print("torchrun rc", r.returncode)

In [ ]:
!cd {SCRIPTS} && python 04_nccl_summary.py